# 01-4. Agent Loop와 ReAct

- 핵심 기술: Plan-Act-Observe-Evaluate 반복 구조, 최대 반복 횟수
- 최종 산출물: 다단계 요청을 스스로 반복 처리하는 Agent Loop

## 1. 학습목표

이 실습을 완료하면 다음을 수행할 수 있다.

1. Agent Loop와 ReAct 구조의 Plan-Act-Observe-Evaluate 단계를 설명할 수 있다.
2. 이전 행동의 관찰 결과를 다음 판단에 반영하는 반복 구조를 구현할 수 있다.
3. 논리적 종료 조건과 안전장치인 최대 반복 횟수의 차이를 구분할 수 있다.
4. 동일 행동이 반복되는 문제를 진단하고 조기 종료 로직으로 수정할 수 있다.

## 2. 문제 상황

Notebook 01-3의 Tool Calling은 한 번의 요청에 대해 Tool을 한 번(또는 한 묶음) 호출하고 끝난다. 
하지만 "128을 4로 나눈 값에 17을 더한 값을 계산해줘"처럼 여러 단계를 거쳐야 하는 요청은 한 번의 Tool 호출만으로 해결되지 않는다. 
첫 번째 계산 결과를 확인한 뒤에야 두 번째로 무엇을 계산해야 할지 알 수 있기 때문이다.  
이런 문제를 해결하려면 이전 행동의 결과(Observation)를 다음 판단(Plan)에 반영하며 여러 차례 반복하는 구조가 필요하다.  

> 한 번의 Tool 호출로 끝나지 않는 요청은, 이전 결과를 다음 판단에 반영하는 반복 구조  
> 없이는 해결할 수 없다.  

## 3. 핵심 개념

### 3.1 Agent Loop와 ReAct란 무엇인가

Agent Loop는 목표를 달성할 때까지 "계획 → 행동 → 관찰 → 평가"를 반복하는 실행 구조이다.  
ReAct(Reasoning + Acting)는 이 반복에서 매 단계마다 "지금까지의 관찰을 바탕으로 다음에  
무엇을 할지 추론(Reasoning)한 뒤 행동(Acting)한다"는 패턴을 가리키는 이름이다.  
즉 ReAct는 Agent Loop를 구현하는 대표적인 방식 중 하나이다.  

### 3.2 개념이 필요한 이유

Tool을 한 번만 호출해서 끝나는 구조로는 다단계 요청을 처리할 수 없다.   
예를 들어 "128 ÷ 4를 구하고 그 결과에 17을 더하라"는 요청은 첫 계산 결과를 알아야 두 번째 계산의 입력을 만들 수 있다.  
Agent Loop는 매 반복마다 지금까지의 관찰 결과를 다음 계획에 반영함으로써 이런 순차적 의존관계를 처리한다.   

### 3.3 주요 구성요소

| 구성요소 | 역할 |
|---|---|
| Plan | 지금까지의 관찰을 바탕으로 다음 행동을 결정 |
| Act | 결정된 행동(Tool 호출 등)을 실제로 실행 |
| Observe | 행동의 결과를 기록 |
| Evaluate | 목표 달성 여부를 판단하고 반복 지속/종료를 결정 |
| 최대 반복 횟수 | 논리적 종료 조건이 실패해도 무한 반복을 막는 안전장치 |

### 3.4 동작 과정

```text
사용자 요청
   ↓
[ Plan: 지금까지의 관찰로 다음 행동 결정 ]
   ↓
[ Act: 행동 실행 ]
   ↓
[ Observe: 결과 기록 → 관찰 목록에 추가 ]
   ↓
[ Evaluate: 완료인가? ] --아니오--> (Plan으로 돌아감, 단 최대 반복 횟수 이내)
   │ 예
   ↓
종료
```

### 3.5 코드와 개념의 대응 관계

| 코드 요소 | 구현 개념 |
|---|---|
| `plan_next_action()` | Plan 단계 |
| `act()` | Act 단계 |
| `state["observations"].append(...)` | Observe 단계 |
| `action["action"] == "finish"` | Evaluate의 논리적 종료 조건 |
| `for _ in range(max_iterations)` | Evaluate의 안전장치(최대 반복 횟수) |

### 3.6 유사 개념과의 차이

| 구분 | 고정 Workflow (Notebook 01-1) | Agent Loop |
|---|---|---|
| 실행 횟수 | 정해진 순서대로 1회 실행 | 목표 달성까지 반복 실행 |
| 다음 단계 결정 | 미리 정해진 코드 순서 | 이전 관찰 결과에 따라 매번 다시 결정 |
| 다단계 의존 요청 처리 | 어려움 | 가능 |
| 종료 시점 | 코드 끝에서 자동 종료 | 논리적 조건 또는 최대 반복 횟수로 종료 |

### 3.7 사용 시점과 적용 조건

이전 단계의 결과를 알아야 다음 단계의 입력을 정할 수 있는, 순차적 의존관계가 있는 다단계 요청에 Agent Loop를 사용한다.   
한 번의 Tool 호출로 끝나는 단순 요청에는 Notebook 01-3의 단일 Tool Calling으로 충분하다.  

### 3.8 한계와 주의사항

- 반복마다 LLM 호출이 발생하므로 단일 호출 구조보다 비용과 지연시간이 늘어난다.
- 논리적 종료 조건(Plan이 "finish"를 반환하는 것)이 제대로 동작하지 않으면 동일한 행동을 반복하며 최대 반복 횟수까지 자원을 낭비할 수 있다.  
- 최대 반복 횟수는 무한 반복을 막는 안전장치일 뿐, 논리적 종료 조건을 대체하지 않는다.  
  안전장치에 의존해 강제 종료된 경우는 "정상 완료"가 아니라 "실패"로 기록해야 한다.

### 3.9 자주 발생하는 오해

"최대 반복 횟수에 도달하면 정상적으로 끝난 것"이라는 생각은 오해이다.  
최대 반복 횟수는 논리적 종료 조건이 실패했을 때 시스템을 보호하는 안전장치이며,   
이 상태로 종료된 실행은 `completed = False`로 기록해 실패로 구분해야 한다.  

### 3.10 핵심 정리

- Agent Loop는 Plan-Act-Observe-Evaluate를 반복하며 다단계 요청을 처리한다.  
- 이전 Observation을 다음 Plan에 반영하지 않으면 동일한 행동이 반복될 수 있다.  
- 논리적 종료 조건과 최대 반복 횟수(안전장치)는 서로 다른 역할을 한다.  
- 반복 횟수가 늘어날수록 비용과 지연시간이 함께 늘어난다는 점을 고려해야 한다.  

## 4. 실행 구조

```text
run_agent_loop(user_request)
   │
   ├─ 1회차: plan_next_action → parse_action → act → observation 기록
   ├─ 2회차: plan_next_action(이전 observation 포함) → ... → observation 기록
   ├─ ...
   └─ finish 행동 또는 max_iterations 도달 시 종료
```

## 5. 환경 설정

반복되는 경로 탐색과 모델 생성 코드는 `src/agentic_ai` 공통 모듈에서 관리한다.  
아래 셀에서는 이 Notebook에 필요한 표준 라이브러리와 공통 기능만 불러온다.  

> 처음 실행하거나 환경 오류가 발생하면 프로젝트 루트의
> `00_environment_check.ipynb`를 먼저 실행한다.


In [1]:
from agentic_ai.config import get_settings
from agentic_ai.logging_utils import save_log
from agentic_ai.models import get_chat_model
from agentic_ai.notebook_utils import print_environment_summary
from agentic_ai.paths import OUTPUT_DIR, PROJECT_ROOT
from agentic_ai.tools import calculate

settings = get_settings()
print_environment_summary(settings, needs_chat_model=True)


[환경 설정 확인]
- 프로젝트: C:\Users\magpi\agentic_ai_lab_202607
- 데이터: C:\Users\magpi\agentic_ai_lab_202607\data
- 출력: C:\Users\magpi\agentic_ai_lab_202607\outputs
- OPENAI_API_KEY: 설정됨
- Chat Model: gpt-4.1-mini


## 6. 최소 실행 예제

Plan 단계 하나만 먼저 실행해서 LLM이 어떤 형식으로 다음 행동을 제안하는지 확인한다.

In [2]:
def plan_next_action(user_request: str, observations: list[dict]) -> str:
    """지금까지의 관찰을 반영해 다음 행동을 한 줄로 요청한다."""
    # 이전 Action과 실행 결과를 다음 Plan의 문맥으로 다시 구성한다.
    history_lines = []
    for i, obs in enumerate(observations, start=1):
        action = obs["action"]
        result = obs["result"]
        value = result.get("output", result.get("error"))
        history_lines.append(f"{i}. calculate({action['action_input']}) -> {value}")
    history_text = "\n".join(history_lines) if history_lines else "(아직 없음)"

    # 출력 형식을 제한해야 다음 단계의 parse_action이 안정적으로 해석할 수 있다.
    prompt = f"""다음 사용자 요청을 해결하기 위한 다음 행동을 한 줄로 알려줘.

사용자 요청: {user_request}

지금까지 관찰 기록:
{history_text}

다음 형식 중 하나로만 답하라.
ACTION: calculate | INPUT: <계산식>
ACTION: finish | INPUT: <최종 답>

규칙:
- 계산값을 직접 추정하지 말고 모든 산술 계산은 calculate 행동으로 수행한다.
- calculate에는 한 번에 하나의 사칙연산만 넣는다.
- 관찰 결과가 충분할 때만 finish를 사용하고, 최종 답은 관찰된 값으로 작성한다."""
    model = get_chat_model()
    response = model.invoke(prompt)
    return response.content


raw = plan_next_action("128을 4로 나눈 값에 17을 더한 값을 계산해줘.", [])
print(raw)

ACTION: calculate | INPUT: 128 / 4


## 7. 단계별 구현

### 7.1 행동 파싱

`plan_next_action()`이 반환한 한 줄 텍스트를 `{"action": ..., "action_input": ...}`
딕셔너리로 변환한다.

In [3]:
def parse_action(raw_text: str) -> dict:
    """'ACTION: x | INPUT: y' 형식의 텍스트를 딕셔너리로 변환한다."""
    # partition은 구분자가 없어도 예외를 내지 않아 빈 값으로 후속 검증을 넘길 수 있다.
    action_part, _, input_part = raw_text.partition("|")
    action = action_part.split(":", 1)[1].strip().lower() if ":" in action_part else ""
    action_input = input_part.split(":", 1)[1].strip() if ":" in input_part else ""
    return {"action": action, "action_input": action_input}


print(parse_action(raw))

{'action': 'calculate', 'action_input': '128 / 4'}


### 7.2 행동 실행 (Act)

이번 실습은 `calculate` Tool 하나만 사용해 반복 구조 자체에 집중한다.  
여러 Tool 중에서 고르는 문제는 Notebook 02-2(Multi-Tool Routing)에서 다룬다.  

In [4]:
def act(action: dict) -> dict:
    """파싱된 행동을 실제로 실행한다."""
    if action["action"] != "calculate":
        return {"status": "error", "error": f"알 수 없는 행동: {action['action']}"}
    try:
        output = calculate(action["action_input"])
        return {"status": "ok", "output": output}
    except ValueError as exc:
        return {"status": "error", "error": str(exc)}


demo_action = parse_action("ACTION: calculate | INPUT: 128 / 4")
print(act(demo_action))

{'status': 'ok', 'output': 32.0}


### 7.3 Agent Loop 조립

**TODO**: `run_agent_loop()`를 작성한다.

- 반환값은 작업지시서가 요구하는 다음 스키마를 따른다:  
  `{"steps": [], "observations": [], "retry_count": 0, "completed": False, "error": None}`  
  (`error`는 안전장치로 강제 종료된 경우 실패 사유를 기록하는 필드다.)  
- 매 반복마다 `retry_count`를 1 증가시킨다.  
- `plan_next_action(user_request, state["observations"])`으로 다음 행동을 얻고  
  `parse_action()`으로 파싱해 `state["steps"]`에 추가한다.  
- 행동이 `"finish"`이면 `state["completed"] = True`, `state["final_answer"]`을 설정하고 반복을 멈춘다.
- 그렇지 않으면 `act()`로 실행하고 결과를 `state["observations"]`에  
  `{"action": action, "result": result}` 형태로 추가한다.  
- `max_iterations`에 도달할 때까지 `finish`가 나오지 않으면 `completed`는 `False`로 남고,   
  `error`에 실패 사유("최대 반복 횟수에 도달했습니다.")를 남긴다.  

예상 출력: `completed=True`이고 `final_answer`가 최종 계산값인 State.

In [5]:
def scripted_math_plan(user_request: str, observations: list[dict]) -> str:
    """Loop 구조를 결정적으로 관찰하기 위한 2단계 계산 Plan."""
    # 관찰 개수에 따라 첫 계산, 두 번째 계산, 종료를 순서대로 반환한다.
    if not observations:
        return "ACTION: calculate | INPUT: 128 / 4"
    if len(observations) == 1:
        first_value = observations[-1]["result"]["output"]
        return f"ACTION: calculate | INPUT: {first_value} + 17"
    return f"ACTION: finish | INPUT: {observations[-1]['result']['output']}"


def run_agent_loop(user_request: str, max_iterations: int = 5, plan_fn=plan_next_action) -> dict:
    """Plan-Act-Observe-Evaluate를 반복하며 요청을 처리한다."""
    # steps는 계획 기록, observations는 실행 결과이며 다음 Plan의 입력으로 다시 사용된다.
    state = {"steps": [], "observations": [], "retry_count": 0, "completed": False, "error": None}

    # 모델이 finish를 선택하지 못해도 max_iterations에서 반드시 멈춘다.
    for _ in range(max_iterations):
        state["retry_count"] += 1
        action = parse_action(plan_fn(user_request, state["observations"]))
        state["steps"].append(action)

        # finish는 Tool을 실행하는 행동이 아니라 Loop를 끝내는 제어 신호다.
        if action["action"] == "finish":
            state["completed"] = True
            state["final_answer"] = action["action_input"]
            break

        # Act 결과를 관찰에 누적해야 다음 Plan이 이전 계산값을 활용할 수 있다.
        result = act(action)
        state["observations"].append({"action": action, "result": result})
    else:
        # for-else의 else는 break 없이 반복 한도를 모두 소진했을 때만 실행된다.
        state["error"] = "최대 반복 횟수에 도달했습니다."

    return state


# 먼저 결정적 Plan으로 Loop 자체를 학습한다. LLM Plan은 아래 확장 실행에서 비교한다.
loop_result = run_agent_loop(
    "128을 4로 나눈 값에 17을 더한 값을 계산해줘.",
    plan_fn=scripted_math_plan,
)
print(loop_result)

llm_loop_result = run_agent_loop("128을 4로 나눈 값에 17을 더한 값을 계산해줘.")
print("LLM Plan 실행:", llm_loop_result)

{'steps': [{'action': 'calculate', 'action_input': '128 / 4'}, {'action': 'calculate', 'action_input': '32.0 + 17'}, {'action': 'finish', 'action_input': '49.0'}], 'observations': [{'action': {'action': 'calculate', 'action_input': '128 / 4'}, 'result': {'status': 'ok', 'output': 32.0}}, {'action': {'action': 'calculate', 'action_input': '32.0 + 17'}, 'result': {'status': 'ok', 'output': 49.0}}], 'retry_count': 3, 'completed': True, 'error': None, 'final_answer': '49.0'}
LLM Plan 실행: {'steps': [{'action': 'calculate', 'action_input': '128 / 4'}, {'action': 'calculate', 'action_input': '32.0 + 17'}, {'action': 'finish', 'action_input': '49.0'}], 'observations': [{'action': {'action': 'calculate', 'action_input': '128 / 4'}, 'result': {'status': 'ok', 'output': 32.0}}, {'action': {'action': 'calculate', 'action_input': '32.0 + 17'}, 'result': {'status': 'ok', 'output': 49.0}}], 'retry_count': 3, 'completed': True, 'error': None, 'final_answer': '49.0'}


## 8. 실행 결과 관찰

`steps`(제안된 행동 전체), `observations`(실제 실행 결과), `retry_count`(반복 횟수),
`completed`(정상 종료 여부)를 함께 확인한다.

In [6]:
print("steps:", loop_result["steps"])
print("observations:", loop_result["observations"])
print("retry_count:", loop_result["retry_count"])
print("completed:", loop_result["completed"])

steps: [{'action': 'calculate', 'action_input': '128 / 4'}, {'action': 'calculate', 'action_input': '32.0 + 17'}, {'action': 'finish', 'action_input': '49.0'}]
observations: [{'action': {'action': 'calculate', 'action_input': '128 / 4'}, 'result': {'status': 'ok', 'output': 32.0}}, {'action': {'action': 'calculate', 'action_input': '32.0 + 17'}, 'result': {'status': 'ok', 'output': 49.0}}]
retry_count: 3
completed: True


**결과 해석**: `steps`에는 매 반복마다 LLM이 제안한 행동이 순서대로 남고,
`observations`에는 그 중 실제로 실행된 행동의 결과만 남는다.  
`retry_count`가 `len(steps)`와 같다면 매 반복마다 정확히 하나의 행동만 제안되었다는 뜻이다.

## 9. 실패 실험: 동일 행동 반복

Plan 단계가 이전 관찰 결과를 무시하고 항상 같은 행동만 제안하면 어떻게 되는지
결정론적으로 재현한다.  
LLM 호출 없이 고정된 함  수로 재현하므로 실행할 때마다 같은
결과를 얻을 수 있다.  

In [7]:
def broken_plan_next_action(user_request: str, observations: list[dict]) -> str:
    """관찰 결과를 무시하고 항상 같은 행동을 제안하는 잘못된 Plan 함수."""
    return "ACTION: calculate | INPUT: 128 / 4"


broken_state = {"steps": [], "observations": [], "retry_count": 0, "completed": False}
# 관찰을 무시하는 Plan은 같은 Action만 누적하고 finish에 도달하지 못한다.
for _ in range(6):
    broken_state["retry_count"] += 1
    action = parse_action(broken_plan_next_action("...", broken_state["observations"]))
    broken_state["steps"].append(action)
    result = act(action)
    broken_state["observations"].append({"action": action, "result": result})

print("retry_count:", broken_state["retry_count"])
print("completed:", broken_state["completed"])
print("고유 행동 종류:", {s["action_input"] for s in broken_state["steps"]})

retry_count: 6
completed: False
고유 행동 종류: {'128 / 4'}


**원인 분석 질문**

- `broken_state["steps"]`에 서로 다른 행동이 몇 종류나 있는가?
- 6번을 반복했지만 왜 `completed`는 여전히 `False`인가?
- 이 상태로 `max_iterations`를 100으로 늘리면 어떤 문제가 새로 생기는가?
  (힌트: 결과가 아니라 비용과 지연시간 관점에서 생각한다.)

## 10. 오류 수정 실습

동일 행동이 반복되는 것을 감지해서, 최대 반복 횟수를 다 채우기 전에 실패로 조기
종료하도록 개선한다.

**TODO**: `detect_repeated_action()`을 작성한다.

- `steps` 리스트의 마지막 두 항목의 `action_input`이 같으면 `True`를 반환한다.
- `steps` 길이가 2 미만이면 `False`를 반환한다.

In [8]:
def detect_repeated_action(steps: list[dict]) -> bool:
    """가장 최근 두 행동이 동일한 입력으로 반복되었는지 확인한다."""
    # 비교할 이전 행동이 없는 첫 단계에서는 반복으로 판단하지 않는다.
    if len(steps) < 2:
        return False
    return steps[-1]["action_input"] == steps[-2]["action_input"]


def run_agent_loop_v2(user_request: str, max_iterations: int = 5, plan_fn=plan_next_action) -> dict:
    """동일 행동 반복을 감지하면 즉시 실패로 중단하는 개선된 Agent Loop."""
    state = {"steps": [], "observations": [], "retry_count": 0, "completed": False, "error": None}

    for _ in range(max_iterations):
        state["retry_count"] += 1
        raw_action = plan_fn(user_request, state["observations"])
        action = parse_action(raw_action)
        state["steps"].append(action)

        if action["action"] == "finish":
            state["completed"] = True
            state["final_answer"] = action["action_input"]
            break

        # 같은 행동을 다시 실행하기 전에 중단해 비용 낭비와 무한 루프를 막는다.
        if detect_repeated_action(state["steps"]):
            state["error"] = "동일 행동이 반복되어 중단했습니다."
            break

        result = act(action)
        state["observations"].append({"action": action, "result": result})
    else:
        # finish도, 반복 감지도 없이 max_iterations를 다 채운 경우에도 실패 사유를 남긴다.
        state["error"] = "최대 반복 횟수에 도달했습니다."

    return state


fixed_broken = run_agent_loop_v2("128을 4로 나눈 값에 17을 더한 값을 계산해줘.", plan_fn=broken_plan_next_action)
print("고장난 Plan으로 실행:", fixed_broken)

fixed_normal = run_agent_loop_v2("128을 4로 나눈 값에 17을 더한 값을 계산해줘.")
print("정상 Plan으로 실행:", fixed_normal)

고장난 Plan으로 실행: {'steps': [{'action': 'calculate', 'action_input': '128 / 4'}, {'action': 'calculate', 'action_input': '128 / 4'}], 'observations': [{'action': {'action': 'calculate', 'action_input': '128 / 4'}, 'result': {'status': 'ok', 'output': 32.0}}], 'retry_count': 2, 'completed': False, 'error': '동일 행동이 반복되어 중단했습니다.'}
정상 Plan으로 실행: {'steps': [{'action': 'calculate', 'action_input': '128 / 4'}, {'action': 'calculate', 'action_input': '32.0 + 17'}, {'action': 'finish', 'action_input': '49.0'}], 'observations': [{'action': {'action': 'calculate', 'action_input': '128 / 4'}, 'result': {'status': 'ok', 'output': 32.0}}, {'action': {'action': 'calculate', 'action_input': '32.0 + 17'}, 'result': {'status': 'ok', 'output': 49.0}}], 'retry_count': 3, 'completed': True, 'error': None, 'final_answer': '49.0'}


**수정 결과 재검증**: 고장난 Plan으로 실행하면 `retry_count`가 2에서 멈추고
`error` 필드가 채워져야 한다(6번을 다 채우지 않는다). 정상 Plan으로 실행하면
`completed=True`와 최종 계산값이 나와야 한다.

In [9]:
assert fixed_broken["retry_count"] == 2
assert fixed_broken["completed"] is False
assert fixed_broken["error"] is not None
assert fixed_normal["completed"] is True
print("재검증 통과")

재검증 통과


## 11. 도전 과제

1. 완료 조건(`action == "finish"`)을 제거하면 어떤 문제가 생기는지 실험해본다.
2. 관찰 결과를 `state["observations"]`에 저장하지 않도록 바꾸면 `plan_next_action()`이
   받는 정보가 어떻게 달라지는지 확인한다.
3. `detect_repeated_action()`의 판단 범위를 마지막 2개가 아니라 마지막 3개로
   넓혀서 더 긴 반복 패턴도 감지하도록 확장해본다.

## 12. 테스트

**테스트 유형: 단위 테스트 — 결정적, 외부 API 호출 없음**

LLM 호출 없이 동작을 검증할 수 있는 함수들을 테스트한다.

In [10]:
assert parse_action("ACTION: calculate | INPUT: 1 + 1") == {"action": "calculate", "action_input": "1 + 1"}
assert act({"action": "calculate", "action_input": "10 / 2"}) == {"status": "ok", "output": 5.0}
assert act({"action": "search_keyword", "action_input": "x"})["status"] == "error"
assert detect_repeated_action(
    [{"action": "calculate", "action_input": "1+1"}, {"action": "calculate", "action_input": "1+1"}]
) is True
assert detect_repeated_action(
    [{"action": "calculate", "action_input": "1+1"}, {"action": "calculate", "action_input": "2+2"}]
) is False
assert detect_repeated_action([{"action": "calculate", "action_input": "1+1"}]) is False
print("테스트 통과")

테스트 통과


## 13. 결과 저장

In [11]:
agent_loop_log = {
    "normal_run": loop_result,
    "broken_plan_demo": broken_state,
    "fixed_broken_plan": fixed_broken,
    "fixed_normal_plan": fixed_normal,
}
saved_path = save_log(agent_loop_log, OUTPUT_DIR / "logs" / "01-4_agent_loop_log.json")
print("저장 위치:", saved_path)

저장 위치: C:\Users\magpi\agentic_ai_lab_202607\outputs\logs\01-4_agent_loop_log.json


## 14. 핵심 정리

- Agent Loop는 Plan-Act-Observe-Evaluate를 반복해 다단계 요청을 처리한다.
- 이전 Observation을 다음 Plan에 반영하지 않으면 동일 행동이 반복될 수 있다.
- 논리적 종료 조건(`finish`)과 안전장치(최대 반복 횟수)는 서로 다른 역할을 하며,
  안전장치로 종료된 실행은 실패로 기록해야 한다.
- 반복 횟수가 늘어날수록 비용과 지연시간이 함께 증가하므로, 조기 종료 로직이
  필요하다.

## 15. 확인 문제

1. Plan, Act, Observe, Evaluate 각 단계가 코드의 어느 부분에 대응하는지 설명하시오.
2. 논리적 종료 조건과 최대 반복 횟수(안전장치)의 차이를 설명하시오.
3. 이번 실습의 실패 실험에서 동일 행동이 반복된 근본 원인은 무엇인가?
4. `detect_repeated_action()`이 없다면 실패 실험은 몇 번 반복 후에 멈추는가?